In [ ]:
# pip install pandas openpyxl biopython requests
# pip install google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 10.2 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd 

df = pd.read_excel("~/Desktop/placenta_sheet.xlsx")

df.head(20)


,GEO Series ID (GSE___),PMID,PMCID,doi (link),Supervisor/Contact/Corresponding author name,Supervisor/Contact/Corresponding author email,Main topic of the publication,"Pregnancy trimester (1st, 2nd, 3rd, term (for full-term delivery), premature (for early delivery due to complications)",Birthweight of offspring provided (yes/no),Gestational Age at delivery provided (yes/no),...,Maternal age at sample collection provided (yes/no),Paternal age at sample collection provided (yes/no),Samples from pregnancy complications collected,Mode of delivery provided (yes/no),Pregnancy complications in data set (list),Fetal complications listed (yes/no),Fetal complications in data set (list),Other Phenotypes Provided (list),Hospital/Center where samples were collected,Country where samples were collected
0,GSE247712,38733496.0,PMC11303482,10.1007/s10456-024-09927-7,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GSE270174,39071344.0,PMC11275976,10.1101/2024.07.12.603155,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GSE232526,39109839.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GSE214007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GSE272695,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,GSE185119,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,GSE247711,38733496.0,PMC11303482,10.1007/s10456-024-09927-7,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,GSE230783,37883444.0,PMC10873279,10.1093/biolre/ioad146,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,GSE248126,38334607.0,PMC10854826,10.3390/cells13030215,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,GSE248093,38334607.0,PMC10854826,10.3390/cells13030215,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [49]:
import os
import re
import time
import json
import pathlib
import pandas as pd
import requests
from typing import Optional, Dict, Any
from Bio import Entrez

# ------------------------------
# CONFIGURATION
# ------------------------------
NCBI_EMAIL = "jjosep31@asu.edu"       # NCBI email
EXCEL_FILE = os.path.expanduser("~/Desktop/placenta_sheet.xlsx")
OUTPUT_DIR = "downloaded_papers"

API_DELAY = 0.3        # polite delay between calls
MAX_RETRIES = 5        # retry attempts for any network call
BASE_BACKOFF = 0.75    # seconds; exponential backoff base
TIMEOUT = 30           # HTTP timeout seconds
USE_PMID_TO_PMCID = True  # set False to never call the converter; use only sheet PMCID

Entrez.email = NCBI_EMAIL

# ------------------------------
# HTTP session + retries
# ------------------------------
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "oa-scraper/1.0 (+https://www.ncbi.nlm.nih.gov/)"})

def backoff_sleep(attempt: int) -> None:
    delay = BASE_BACKOFF * (2 ** (attempt - 1)) + (0.05 * (attempt - 1))
    time.sleep(delay)

def retrying_get(url: str, *, stream: bool=False, expected_status: int=200) -> Optional[requests.Response]:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = SESSION.get(url, timeout=TIMEOUT, allow_redirects=True, stream=stream)
            if r.status_code == expected_status:
                return r
            if r.status_code in (429, 500, 502, 503, 504):
                last_err = RuntimeError(f"HTTP {r.status_code}")
            else:
                return None
        except requests.RequestException as e:
            last_err = e
        if attempt < MAX_RETRIES:
            backoff_sleep(attempt)
    return None

def retrying_entrez_efetch(db: str, id_: str, rettype: str, retmode: str) -> Optional[str]:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            handle = Entrez.efetch(db=db, id=id_, rettype=rettype, retmode=retmode)
            txt = handle.read()
            handle.close()
            if isinstance(txt, bytes):
                txt = txt.decode("utf-8", errors="ignore")
            if txt:
                return txt
            last_err = RuntimeError("Empty efetch body")
        except Exception as e:
            last_err = e
        if attempt < MAX_RETRIES:
            backoff_sleep(attempt)
    return None

# ------------------------------
# Utils
# ------------------------------
def ensure_dir(path: str) -> None:
    pathlib.Path(path).mkdir(parents=True, exist_ok=True)

def sanitize_filename(s: str) -> str:
    s = s.strip().replace(" ", "_")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)

def save_binary(content: bytes, path: str) -> None:
    with open(path, "wb") as f:
        f.write(content)

def save_text(content: str, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

def as_clean_str(x) -> Optional[str]:
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None

def is_jats_xml(xml_text: str) -> bool:
    if not xml_text:
        return False
    lower = xml_text.lower()
    return ("<article" in lower) and ("<body" in lower or "journal-meta" in lower or "article-meta" in lower)

# ------------------------------
# ID mapping via PMC ID Converter
# ------------------------------
def pmcid_from_pmid_via_idconv(pmid: str) -> Optional[str]:
    url = (
        "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
        f"?format=json&tool=script&email={NCBI_EMAIL}&ids={pmid}"
    )
    r = retrying_get(url)
    if not r:
        return None
    try:
        data = r.json()
    except json.JSONDecodeError:
        return None
    recs = data.get("records", [])
    if not recs:
        return None
    rec = recs[0]
    pmcid = rec.get("pmcid")
    return pmcid if pmcid else None

# ------------------------------
# PMC (Entrez)
# ------------------------------
def fetch_pmc_xml_via_entrez(pmcid: str) -> Optional[str]:
    xml_text = retrying_entrez_efetch(db="pmc", id_=pmcid, rettype="xml", retmode="text")
    if xml_text and is_jats_xml(xml_text):
        return xml_text
    return None

def download_via_pmc(pmcid: str, out_dir: str) -> Optional[str]:
    print(f"  > PMC: trying JATS XML for {pmcid} ...")
    xml_text = fetch_pmc_xml_via_entrez(pmcid)
    if xml_text:
        out = os.path.join(out_dir, f"{sanitize_filename(pmcid)}.xml")
        save_text(xml_text, out)
        print(f"    OK PMC XML: {out}")
        return out
    print("    No PMC XML.")
    return None

# ------------------------------
# Europe PMC
# ------------------------------
def europe_pmc_search(pmid: Optional[str], pmcid: Optional[str]) -> Optional[Dict[str, Any]]:
    base = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    # Try PMCID first, then PMID
    queries = []
    if pmcid:
        queries.append(f"EXT_ID:{pmcid}")
    if pmid:
        queries.append(f"EXT_ID:{pmid}")

    for q in queries:
        url = f"{base}?query={q}&format=json"
        r = retrying_get(url)
        time.sleep(API_DELAY)
        if not r:
            continue
        try:
            data = r.json()
        except json.JSONDecodeError:
            continue
        results = (data.get("resultList") or {}).get("result") or []
        if results:
            return results[0]
    return None

def fetch_europe_pmc_fulltext_xml(pmcid: str) -> Optional[bytes]:
    url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML"
    r = retrying_get(url)
    if r and r.text and is_jats_xml(r.text):
        return r.content
    return None

def pick_epmc_fulltext_url(result: Dict[str, Any]) -> Optional[str]:
    ft = result.get("fullTextUrlList", {})
    urls = ft.get("fullTextUrl", []) if isinstance(ft, dict) else []
    pdf_oa = [u for u in urls if u.get("documentStyle","").lower()=="pdf" and "open" in u.get("availability","").lower()]
    if pdf_oa:
        return pdf_oa[0].get("url")
    pdf_any = [u for u in urls if u.get("documentStyle","").lower()=="pdf"]
    if pdf_any:
        return pdf_any[0].get("url")
    html_oa = [u for u in urls if u.get("documentStyle","").lower()=="html" and "open" in u.get("availability","").lower()]
    if html_oa:
        return html_oa[0].get("url")
    if urls:
        return urls[0].get("url")
    return None

def download_via_europe_pmc(pmcid: Optional[str], pmid: Optional[str], out_dir: str) -> Optional[str]:
    # 1) If PMCID, try EPMC XML
    if pmcid:
        print(f"  > Europe PMC: trying XML for {pmcid} ...")
        xml_bytes = fetch_europe_pmc_fulltext_xml(pmcid)
        if xml_bytes:
            out = os.path.join(out_dir, f"{sanitize_filename(pmcid)}.xml")
            save_binary(xml_bytes, out)
            print(f"    OK EPMC XML: {out}")
            return out
        print("    No Europe PMC XML for this PMCID.")

    # 2) Otherwise search for any OA link using PMCID/PMID
    print("  > Europe PMC: searching for OA links ...")
    result = europe_pmc_search(pmid=pmid, pmcid=pmcid)
    if not result:
        print("    No Europe PMC record.")
        return None

    url = pick_epmc_fulltext_url(result)
    if not url:
        print("    No full-text URL in Europe PMC record.")
        return None

    print(f"    Europe PMC: downloading {url}")
    r = retrying_get(url)
    if not r or not r.content:
        print("    Europe PMC download failed.")
        return None

    ct = (r.headers.get("Content-Type") or "").lower()
    ext = ".pdf" if ("pdf" in ct or url.lower().endswith(".pdf")) else ".html"
    base = pmcid or pmid or "paper"
    out = os.path.join(out_dir, f"{sanitize_filename(base)}{ext}")
    save_binary(r.content, out)
    print(f"    OK EPMC full text: {out}")
    return out

# ------------------------------
# Orchestration per row (PMC + Europe PMC only)
# ------------------------------
def process_row(pmcid: Optional[str], pmid: Optional[str]) -> Dict[str, Any]:
    outcome = {
        "pmcid": pmcid,
        "pmid": pmid,
        "saved_path": None,
        "source": None,
        "format": None,   # xml/pdf/html
        "status": "failed",
        "notes": ""
    }

    # Normalize PMCID
    if pmcid:
        pmcid = pmcid.strip().upper().replace(" ", "")
        if pmcid.startswith("PMCPMC"):
            pmcid = pmcid.replace("PMCPMC", "PMC", 1)
        if not pmcid.startswith("PMC") and pmcid.isdigit():
            pmcid = "PMC" + pmcid
        outcome["pmcid"] = pmcid

    # If we have PMID but no PMCID, optionally map PMID->PMCID
    if not pmcid and pmid and USE_PMID_TO_PMCID:
        conv = pmcid_from_pmid_via_idconv(pmid)
        time.sleep(API_DELAY)
        if conv:
            print(f"    PMID {pmid} → PMCID {conv}")
            pmcid = conv
            outcome["pmcid"] = pmcid
        else:
            outcome["notes"] += "PMCID conversion failed; "

    # 1) PMC XML
    if pmcid:
        path = download_via_pmc(pmcid, OUTPUT_DIR)
        time.sleep(API_DELAY)
        if path:
            outcome.update({"saved_path": path, "source": "PMC", "format": "xml", "status": "ok"})
            return outcome

    # 2) Europe PMC (XML if PMCID, else OA links via PMCID/PMID)
    path = download_via_europe_pmc(pmcid=pmcid, pmid=pmid, out_dir=OUTPUT_DIR)
    time.sleep(API_DELAY)
    if path:
        fmt = "xml" if path.lower().endswith(".xml") else ("pdf" if path.lower().endswith(".pdf") else "html")
        outcome.update({"saved_path": path, "source": "Europe PMC", "format": fmt, "status": "ok"})
        return outcome

    outcome["notes"] += "No OA via PMC/Europe PMC."
    return outcome

# ------------------------------
# Main
# ------------------------------
def main():
    ensure_dir(OUTPUT_DIR)
    print(f"Loading data from {EXCEL_FILE} ...")
    df = pd.read_excel(EXCEL_FILE)

    # Normalize column names
    df.columns = df.columns.str.strip().str.lower()

    # Replace empty strings with NaN
    df = df.replace(r"^\s*$", pd.NA, regex=True)

    # Keep rows with at least one identifier (PMCID or PMID)
    need_cols = [c for c in ["pmcid", "pmid"] if c in df.columns]
    if need_cols:
        df = df.dropna(subset=need_cols, how="all")

    total = len(df)
    print(f"Found {total} rows (after cleaning).")

    results = []
    for i, row in df.iterrows():
        print(f"\nProcessing {i+1} / {total} ...")
        pmcid = as_clean_str(row.get("pmcid"))
        pmid  = as_clean_str(row.get("pmid"))

        # Normalize PMID like "000123" -> "123"
        if pmid and pmid.isdigit():
            pmid = str(int(pmid))

        print(f"    IDs present -> PMCID:{bool(pmcid)} PMID:{bool(pmid)}")

        outcome = process_row(pmcid=pmcid, pmid=pmid)
        results.append(outcome)

    # Save detailed log (all rows)
    summary_csv = os.path.join(OUTPUT_DIR, "download_summary.csv")
    results_df = pd.DataFrame(results)
    results_df.to_csv(summary_csv, index=False)

    # Split success/failure
    ok_df   = results_df[results_df["status"] == "ok"].copy()
    fail_df = results_df[results_df["status"] != "ok"].copy()

    # Save convenience CSVs
    successes_csv = os.path.join(OUTPUT_DIR, "download_successes.csv")
    failures_csv  = os.path.join(OUTPUT_DIR, "download_failures.csv")
    ok_df.to_csv(successes_csv, index=False)
    fail_df.to_csv(failures_csv,  index=False)

    # Counts by source
    by_source = ok_df["source"].value_counts().to_dict() if not ok_df.empty else {}
    n_pmc  = by_source.get("PMC", 0)
    n_epmc = by_source.get("Europe PMC", 0)

    # Formats
    by_format = ok_df["format"].value_counts().to_dict() if not ok_df.empty else {}

    print("\n--- Summary ---")
    print(f"Total rows processed:   {len(results_df)}")
    print(f"Succeeded (all):        {len(ok_df)}")
    print(f"  - from PMC:           {n_pmc}")
    print(f"  - from Europe PMC:    {n_epmc}")
    print(f"Failed:                 {len(fail_df)}")

    if by_format:
        print("\nFormats among successes:")
        for k, v in by_format.items():
            print(f"  {k.upper():<5}: {v}")

    if not fail_df.empty:
        print("\nExamples of failures (up to 5):")
        for _, r in fail_df.head(5).iterrows():
            print(f"  pmcid={r.get('pmcid')}, pmid={r.get('pmid')} -> {r.get('notes')}")

    print(f"\nAll rows CSV:     {summary_csv}")
    print(f"Successes CSV:    {successes_csv}")
    print(f"Failures CSV:     {failures_csv}")
    print(f"Files saved in:   {os.path.abspath(OUTPUT_DIR)}")

if __name__ == "__main__":
    main()

Loading data from /Users/jjoseph/Desktop/placenta_sheet.xlsx ...
Found 1382 rows (after cleaning).

Processing 1 / 1382 ...
    IDs present -> PMCID:True PMID:True
  > PMC: trying JATS XML for PMC11303482 ...
    OK PMC XML: downloaded_papers/PMC11303482.xml

Processing 2 / 1382 ...
    IDs present -> PMCID:True PMID:True
  > PMC: trying JATS XML for PMC11275976 ...
    OK PMC XML: downloaded_papers/PMC11275976.xml

Processing 3 / 1382 ...
    IDs present -> PMCID:False PMID:True
  > Europe PMC: searching for OA links ...
    No Europe PMC record.

Processing 7 / 1382 ...
    IDs present -> PMCID:True PMID:True
  > PMC: trying JATS XML for PMC11303482 ...
    OK PMC XML: downloaded_papers/PMC11303482.xml

Processing 8 / 1382 ...
    IDs present -> PMCID:True PMID:True
  > PMC: trying JATS XML for PMC10873279 ...
    OK PMC XML: downloaded_papers/PMC10873279.xml

Processing 9 / 1382 ...
    IDs present -> PMCID:True PMID:True
  > PMC: trying JATS XML for PMC10854826 ...
    OK PMC XML: 

In [61]:
import os
import xml.etree.ElementTree as ET
import json

# --- CONFIGURATION ---
# This script will read from the folder created by fetch_papers.py
INPUT_DIR = "downloaded_papers"

# This is the name of the output file it will create
OUTPUT_JSON_FILE = "processed_papers.json"

# Settings for how to chunk the text
CHUNK_SIZE = 5000 # Characters per chunk (roughly 1000-1200 words)
OVERLAP_SIZE = 500  # Characters to overlap between chunks to avoid cutting sentences
# --------------------


def extract_text_from_xml(xml_file_path):
    """Parses an XML file and extracts all human-readable text."""
    try:
        tree = ET.parse(xml_file_path)
        root = tree.getroot()
        all_text = ' '.join(node.text for node in root.iter() if node.text)
        cleaned_text = ' '.join(all_text.split())
        return cleaned_text
    except ET.ParseError as e:
        print(f"    > Could not parse XML file {os.path.basename(xml_file_path)}. Error: {e}")
        return None
    except Exception as e:
        print(f"    > An unexpected error occurred with file {os.path.basename(xml_file_path)}: {e}")
        return None


def split_text_into_chunks(text):
    """Splits a long text into smaller, overlapping chunks."""
    if not text:
        return []
        
    chunks = []
    start = 0
    while start < len(text):
        end = start + CHUNK_SIZE
        chunks.append(text[start:end])
        start += CHUNK_SIZE - OVERLAP_SIZE
        
    return chunks

# --- MAIN SCRIPT ---
all_papers_data = []

print(f"Starting to process files from the '{INPUT_DIR}' directory...")
xml_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.xml')]

for index, filename in enumerate(xml_files):
    print(f"\nProcessing file {index + 1}/{len(xml_files)}: {filename}...")
    file_path = os.path.join(INPUT_DIR, filename)
    
    full_text = extract_text_from_xml(file_path)
    
    if full_text:
        text_chunks = split_text_into_chunks(full_text)
        print(f"  > Extracted text and created {len(text_chunks)} chunks.")
        
        pmcid = filename.replace('.xml', '')
        
        all_papers_data.append({
            "pmcid": pmcid,
            "chunks": text_chunks
        })

with open(OUTPUT_JSON_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_papers_data, f, indent=4)

print("\n--- Process Complete ---")
print(f"Successfully processed {len(all_papers_data)} papers.")
print(f"All chunked text has been saved to '{OUTPUT_JSON_FILE}'.")

Starting to process files from the 'downloaded_papers' directory...

Processing file 1/850: PMC10861538.xml...
  > Extracted text and created 10 chunks.

Processing file 2/850: PMC11220921.xml...
  > Extracted text and created 1 chunks.

Processing file 3/850: PMC10288560.xml...
  > Extracted text and created 6 chunks.

Processing file 4/850: PMC9203278.xml...
  > Extracted text and created 13 chunks.

Processing file 5/850: PMC7156023.xml...
  > Extracted text and created 2 chunks.

Processing file 6/850: PMC3534415.xml...
  > Extracted text and created 6 chunks.

Processing file 7/850: PMC3509780.xml...
  > Extracted text and created 1 chunks.

Processing file 8/850: PMC10506314.xml...
  > Extracted text and created 11 chunks.

Processing file 9/850: PMC6824492.xml...
  > Extracted text and created 1 chunks.

Processing file 10/850: PMC4248291.xml...
  > Extracted text and created 8 chunks.

Processing file 11/850: PMC1175992.xml...
  > Extracted text and created 6 chunks.

Processin

Starting analysis of 850 papers.

Processing paper 1/850 (PMCID: PMC10861538)...
    > Successfully extracted data for PMC10861538

Processing paper 2/850 (PMCID: PMC11220921)...
    > Successfully extracted data for PMC11220921

Processing paper 3/850 (PMCID: PMC10288560)...
    > Successfully extracted data for PMC10288560

Processing paper 4/850 (PMCID: PMC9203278)...
    > Successfully extracted data for PMC9203278

Processing paper 5/850 (PMCID: PMC7156023)...
    > Successfully extracted data for PMC7156023

Processing paper 6/850 (PMCID: PMC3534415)...
    > Successfully extracted data for PMC3534415

Processing paper 7/850 (PMCID: PMC3509780)...
    > Successfully extracted data for PMC3509780

Processing paper 8/850 (PMCID: PMC10506314)...
    > Successfully extracted data for PMC10506314

Processing paper 9/850 (PMCID: PMC6824492)...
    > Successfully extracted data for PMC6824492

Processing paper 10/850 (PMCID: PMC4248291)...
    > Successfully extracted data for PMC424829

In [ ]:
import re
from difflib import SequenceMatcher
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import pandas as pd


# ========= USER SETTINGS =========
STUDENT_XLSX = "students.xlsx"     # <-- path to the Student spreadsheet
AI_XLSX      = "ai.xlsAnx"           # <-- path to the AI spreadsheet
OUTPUT_XLSX  = "ai_vs_students_comparison.xlsx"
# =================================


# Canonical field names (exactly the set you provided, fixed spelling)
CANON_QUESTIONS = [
    "Pregnancy trimester (1st, 2nd, 3rd, term (for full-term delivery), premature (for early delivery due to complications))",
    "Birthweight of offspring provided (yes/no)",
    "Gestational Age at delivery provided (yes/no)",
    "Gestational Age at sample collection provided (yes/no)",
    "Sex of Offspring Provided (yes/no)",
    "Parity provided (yes/no)",
    "Gravidity provided (yes/no)",
    "Number of offspring per pregnancy provided (yes/no)",
    "Self-reported race/ethnicity of mother provided (yes/no)",
    "Genetic ancestry or genetic strain provided (yes/no)",
    "Maternal Height provided (yes/no)",
    "Maternal Pre-pregnancy Weight provided (yes/no)",
    "Paternal Height provided (yes/no)",
    "Paternal Weight provided (yes/no)",
    "Maternal age at sample collection provided (yes/no)",
    "Paternal age at sample collection provided (yes/no)",
    "Samples from pregnancy complications collected",
    "Mode of delivery provided (yes/no)",
    "Pregnancy complications in data set (list)",
    "Fetal complications listed (yes/no)",
    "Fetal complications in data set (list)",
    "Other Phenotypes Provided (list)",
    "Hospital/Center where samples were collected",
    "Country where samples were collected",
]

# Heuristics: which of the above should be treated like yes/no
YESNO_FIELDS = {
    q for q in CANON_QUESTIONS
    if ("(yes/no)" in q.lower()) or ("listed" in q.lower()) or ("collected" in q.lower())
}

# The “trimester” field is categorical; we normalize to one of:
TRIMESTER_FIELD = CANON_QUESTIONS[0]
TRIMESTER_CANON = {
    "1st": {"1", "first", "1st"},
    "2nd": {"2", "second", "2nd"},
    "3rd": {"3", "third", "3rd"},
    "term": {"term", "full-term", "full term"},
    "premature": {"preterm", "premature", "early"},
}

# Column header matcher:
#  - keys are the canonical questions
#  - values are lists of regex patterns that should match a column name (case-insensitive)
# These include common misspellings seen in your sheet (“Pregnoncy”, “Maternol”, etc.)
HEADER_PATTERNS: Dict[str, List[str]] = {
    TRIMESTER_FIELD: [
        r"pregn[ae]n?cy\s*trimester", r"trimester"
    ],
    "Birthweight of offspring provided (yes/no)": [
        r"birth[-\s]*weight.*provided", r"birthweight.*provided"
    ],
    "Gestational Age at delivery provided (yes/no)": [
        r"gestation[aol]*\s*age.*delivery.*provided"
    ],
    "Gestational Age at sample collection provided (yes/no)": [
        r"gestation[aol]*\s*age.*sample.*collection.*provided"
    ],
    "Sex of Offspring Provided (yes/no)": [
        r"sex.*offspring.*provided"
    ],
    "Parity provided (yes/no)": [
        r"parity.*provided"
    ],
    "Gravidity provided (yes/no)": [
        r"gravidity.*provided"
    ],
    "Number of offspring per pregnancy provided (yes/no)": [
        r"number.*offspring.*pregn[ae]n?cy.*provided"
    ],
    "Self-reported race/ethnicity of mother provided (yes/no)": [
        r"(self[-\s]*reported.*race|ethnicity).*mother.*provided"
    ],
    "Genetic ancestry or genetic strain provided (yes/no)": [
        r"(genetic\s*ancestry|genetic\s*strain).*provided"
    ],
    "Maternal Height provided (yes/no)": [
        r"mater?n?ol\s*height.*provided", r"maternal\s*height.*provided"
    ],
    "Maternal Pre-pregnancy Weight provided (yes/no)": [
        r"mater?n?ol\s*pre[-\s]*pregn[ae]n?cy\s*weight.*provided"
    ],
    "Paternal Height provided (yes/no)": [
        r"pater?n?ol\s*height.*provided", r"paternal\s*height.*provided"
    ],
    "Paternal Weight provided (yes/no)": [
        r"pater?n?ol\s*weight.*provided", r"paternal\s*weight.*provided"
    ],
    "Maternal age at sample collection provided (yes/no)": [
        r"mater?n?ol\s*age.*sample.*collection.*provided"
    ],
    "Paternal age at sample collection provided (yes/no)": [
        r"pater?n?ol\s*age.*sample.*collection.*provided"
    ],
    "Samples from pregnancy complications collected": [
        r"samples.*pregn[ae]n?cy.*complications.*collected"
    ],
    "Mode of delivery provided (yes/no)": [
        r"mode.*delivery.*provided"
    ],
    "Pregnancy complications in data set (list)": [
        r"pregn[ae]n?cy.*complications.*(in\s*data\s*set|list)"
    ],
    "Fetal complications listed (yes/no)": [
        r"fetal.*complications.*listed"
    ],
    "Fetal complications in data set (list)": [
        r"fetal.*complications.*(in\s*data\s*set|list)"
    ],
    "Other Phenotypes Provided (list)": [
        r"other.*phenotypes.*provided"
    ],
    "Hospital/Center where samples were collected": [
        r"(hospital|center|centre).*(samples|collected)"
    ],
    "Country where samples were collected": [
        r"country.*(samples|collected)"
    ],
}

# GEO Series ID column detection patterns
GEO_ID_PATTERNS = [
    r"^geo\s*series\s*id", r"^geo\s*id$", r"^series\s*id$", r"\(gse", r"\bgse\d{3,}\b"
]


def _clean(s: Optional[str]) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return str(s).strip()


def find_best_col(df: pd.DataFrame, patterns: List[str]) -> Optional[str]:
    """Find the first column whose name matches any pattern (case-insensitive)."""
    cols = list(df.columns)
    for col in cols:
        name = str(col).lower()
        for pat in patterns:
            if re.search(pat, name, flags=re.IGNORECASE):
                return col
    return None


def build_colmap(df: pd.DataFrame) -> Dict[str, Optional[str]]:
    """
    Build a mapping from canonical field name -> actual column name in df.
    Uses HEADER_PATTERNS and falls back to fuzzy name matching if needed.
    """
    colmap: Dict[str, Optional[str]] = {}
    for canon, pats in HEADER_PATTERNS.items():
        found = find_best_col(df, pats)
        if found is None:
            # fuzzy fallback: choose the column with highest token overlap
            best_col, best_score = None, 0.0
            canon_tokens = set(re.findall(r"[a-z]+", canon.lower()))
            for c in df.columns:
                tokens = set(re.findall(r"[a-z]+", str(c).lower()))
                inter = len(canon_tokens & tokens)
                score = inter / max(1, len(canon_tokens))
                if score > best_score:
                    best_col, best_score = c, score
            # only accept if overlap is decent
            colmap[canon] = best_col if best_score >= 0.35 else None
        else:
            colmap[canon] = found
    return colmap


def normalize_yesno(val: str) -> str:
    s = _clean(val).lower()
    if "yes" in s:
        return "Yes"
    return "No"


def normalize_trimester(val: str) -> str:
    s = _clean(val).lower()
    if not s:
        return ""
    # pull out keywords; try exact token matches first
    for canon, keys in TRIMESTER_CANON.items():
        for k in keys:
            if re.search(rf"\b{k}\b", s):
                return canon
    # numeric-only like "1", "2", "3"
    if re.fullmatch(r"[123]", s):
        return {"1": "1st", "2": "2nd", "3": "3rd"}[s]
    return s  # leave as-is (will be compared fuzzily)


def token_set(s: str) -> set:
    return set(re.findall(r"[a-z0-9]+", _clean(s).lower()))


def text_match(a: str, b: str, sim_threshold: float = 0.80, jaccard_threshold: float = 0.40) -> Tuple[bool, float, float]:
    """
    Compare two free-text values in a tolerant way:
      - token Jaccard overlap
      - normalized SequenceMatcher ratio
    Returns (is_match, jaccard, seq_ratio)
    """
    ta, tb = token_set(a), token_set(b)
    if not ta and not tb:
        return True, 1.0, 1.0
    if not ta or not tb:
        # if one is empty but the other says "no", count as match for yes/no-like texts
        pass
    inter = len(ta & tb)
    union = len(ta | tb) or 1
    jacc = inter / union
    seq = SequenceMatcher(None, _clean(a).lower(), _clean(b).lower()).ratio()
    return (jacc >= jaccard_threshold or seq >= sim_threshold), jacc, seq


def load_sheet(path: str) -> pd.DataFrame:
    # Support both Excel and CSV
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    return pd.read_excel(path)


def detect_geo_col(df: pd.DataFrame) -> str:
    col = find_best_col(df, GEO_ID_PATTERNS)
    if col:
        return col
    # fallback: choose a column that looks like it contains GSE ids
    for c in df.columns:
        sample_vals = df[c].astype(str).head(50).tolist()
        if any(re.search(r"\bGSE\d{3,}\b", v, flags=re.IGNORECASE) for v in sample_vals):
            return c
    raise ValueError("Could not detect GEO Series ID column.")


def normalize_field_value(field: str, val: str) -> str:
    if field in YESNO_FIELDS:
        return normalize_yesno(val)
    if field == TRIMESTER_FIELD:
        return normalize_trimester(val)
    return _clean(val)


def compare_rows(student_row: pd.Series,
                 ai_row: pd.Series,
                 student_map: Dict[str, Optional[str]],
                 ai_map: Dict[str, Optional[str]]) -> Dict[str, dict]:
    """
    Return: dict[field] = {
        'student': value,
        'ai': value,
        'match': bool,
        'details': str
    }
    """
    out = {}
    for field in CANON_QUESTIONS:
        scol = student_map.get(field)
        acol = ai_map.get(field)

        s_val_raw = "" if not scol else student_row.get(scol, "")
        a_val_raw = "" if not acol else ai_row.get(acol, "")

        s_val = normalize_field_value(field, s_val_raw)
        a_val = normalize_field_value(field, a_val_raw)

        if field in YESNO_FIELDS:
            match = (s_val == a_val)
            detail = "yes/no normalized"
        elif field == TRIMESTER_FIELD:
            # allow fuzzy if not in canonical set
            if s_val in TRIMESTER_CANON or s_val in {"1st", "2nd", "3rd", "term", "premature"}:
                if a_val in TRIMESTER_CANON or a_val in {"1st", "2nd", "3rd", "term", "premature"}:
                    match = (s_val == a_val)
                else:
                    # fuzzy compare canonical to raw ai value
                    match, j, r = text_match(s_val, a_val, sim_threshold=0.80, jaccard_threshold=0.34)
                    detail = f"trimester fuzzy jacc={j:.2f} seq={r:.2f}"
                    out[field] = {"student": s_val, "ai": a_val, "match": match, "details": detail}
                    continue
            else:
                match, j, r = text_match(s_val, a_val, sim_threshold=0.80, jaccard_threshold=0.34)
            detail = "trimester normalized/fuzzy"
        else:
            match, j, r = text_match(s_val, a_val)
            detail = f"text fuzzy jacc={j:.2f} seq={r:.2f}"

        out[field] = {
            "student": s_val,
            "ai": a_val,
            "match": match,
            "details": detail
        }
    return out


def main():
    # Load
    student_df = load_sheet(STUDENT_XLSX)
    ai_df = load_sheet(AI_XLSX)

    # Detect GEO id columns
    s_geo_col = detect_geo_col(student_df)
    a_geo_col = detect_geo_col(ai_df)

    # Build header mappings
    s_map = build_colmap(student_df)
    a_map = build_colmap(ai_df)

    # Keep only relevant columns for speed
    s_keep = [s_geo_col] + [c for c in s_map.values() if c is not None]
    a_keep = [a_geo_col] + [c for c in a_map.values() if c is not None]
    student_df_small = student_df[s_keep].copy()
    ai_df_small = ai_df[a_keep].copy()

    # Standardize GEO key
    def norm_gse(x):
        sx = _clean(x).upper()
        m = re.search(r"(GSE\d{3,})", sx)
        return m.group(1) if m else sx

    student_df_small["GEO_KEY"] = student_df_small[s_geo_col].map(norm_gse)
    ai_df_small["GEO_KEY"] = ai_df_small[a_geo_col].map(norm_gse)

    # Deduplicate on GEO_KEY (keep first); you can switch to grouping if needed
    student_df_small = student_df_small.drop_duplicates(subset=["GEO_KEY"])
    ai_df_small = ai_df_small.drop_duplicates(subset=["GEO_KEY"])

    # Merge on GEO key (inner join = only studies seen by both)
    

    # Per-row comparison
    # Per-row comparison
    records = []
    field_agree = defaultdict(lambda: {"n": 0, "agree": 0})

    def with_suffix(colname: Optional[str], suffix: str) -> Optional[str]:
        if not colname:
            return None
        return f"{colname}{suffix}"

    # Map canonical fields -> suffixed columns in the merged frame
    s_map_suff = {k: with_suffix(v, "_stu") for k, v in s_map.items()}
    a_map_suff = {k: with_suffix(v, "_ai")  for k, v in a_map.items()}

    # Optional sanity check: warn about any columns we couldn't find post-merge
    missing_student = [c for c in s_map_suff.values() if c and c not in merged.columns]
    missing_ai      = [c for c in a_map_suff.values() if c and c not in merged.columns]
    if missing_student or missing_ai:
        print("WARN: some mapped columns not found after merge.")
        if missing_student: print("  Missing student cols:", missing_student)
        if missing_ai:      print("  Missing AI cols:", missing_ai)

    for _, row in merged.iterrows():
        # Compare using the merged row for both sides; the maps pick the right columns
        cmpd = compare_rows(
            student_row=row,            # merged row (use *_stu via s_map_suff)
            ai_row=row,                 # merged row (use *_ai  via a_map_suff)
            student_map=s_map_suff,
            ai_map=a_map_suff
        )

        base = {"GEO_ID": row["GEO_KEY"]}
        for field in CANON_QUESTIONS:
            cell = cmpd[field]
            base[f"{field} — Student"] = cell["student"]
            base[f"{field} — AI"]      = cell["ai"]
            base[f"{field} — Match"]   = "Yes" if cell["match"] else "No"
            base[f"{field} — Notes"]   = cell["details"]

            field_agree[field]["n"]     += 1
            field_agree[field]["agree"] += int(cell["match"])

        records.append(base)

    per_study_df = pd.DataFrame.from_records(records)


    # Field-level summary table
    summary_rows = []
    for field in CANON_QUESTIONS:
        n = field_agree[field]["n"]
        k = field_agree[field]["agree"]
        acc = (k / n) if n else 0.0
        summary_rows.append({
            "Field": field,
            "Compared": n,
            "Matches": k,
            "AgreementRate": round(acc, 4)
        })
    summary_df = pd.DataFrame(summary_rows).sort_values("AgreementRate", ascending=False)

    # Save outputs
    with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as xl:
        per_study_df.to_excel(xl, index=False, sheet_name="PerStudyComparison")
        summary_df.to_excel(xl, index=False, sheet_name="FieldSummary")

    print(f"Done. Wrote: {OUTPUT_XLSX}")
    print("Sheets:")
    print(" - PerStudyComparison: per GSE, side-by-side values and match flags")
    print(" - FieldSummary: counts and agreement rates per field")
    return per_study_df, summary_df



if __name__ == "__main__":
    main()


/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/_read_only.py:81: UserWarning: Cell I400 is marked as a date but the serial value 3698034.0 is outside the limits for dates. The cell will be treated as an error.
  for idx, row in parser.parse():
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/_read_only.py:81: UserWarning: Cell F650 is marked as a date but the serial value 200138762.0 is outside the limits for dates. The cell will be treated as an error.
  for idx, row in parser.parse():
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/_read_only.py:81: UserWarning: Cell F744 is marked as a date but the serial value 3698036.0 is outside the limits for dates. The cell will be treated as an error.
  for idx, row in parser.parse():


NameError: name 'merged' is not defined

In [27]:
import matplotlib.pyplot as plt
import seaborn as sns

def make_figures(per_study_df, summary_df, output_prefix="comparison_figs"):
    # --- 1. Agreement Rate by Field ---
    plt.figure(figsize=(8, 10))
    sns.barplot(data=summary_df, y="Field", x="AgreementRate", palette="viridis")
    plt.title("Agreement Rate by Metadata Field")
    plt.xlabel("Agreement Rate")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_agreement_by_field.png", dpi=300)
    plt.close()

    # --- 2. Confusion Matrix for Yes/No ---
    yesno_cols = [c for c in per_study_df.columns if "— Match" in c and "(yes/no)" in c.lower()]
    cm_data = []
    for col in yesno_cols:
        field = col.split(" — ")[0]
        stu_col = f"{field} — Student"
        ai_col  = f"{field} — AI"
        tmp = per_study_df[[stu_col, ai_col]].copy()
        tmp.columns = ["Student", "AI"]
        cm_data.append(tmp)
    if cm_data:
        big = pd.concat(cm_data)
        cm = pd.crosstab(big["Student"], big["AI"])
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title("AI vs Student (Yes/No Fields Combined)")
        plt.savefig(f"{output_prefix}_yesno_confusion.png", dpi=300)
        plt.close()

    # --- 3. Per-Study Agreement Histogram ---
    match_cols = [c for c in per_study_df.columns if "— Match" in c]
    per_study_df["AgreementRate"] = (
        per_study_df[match_cols].apply(lambda row: sum(v=="Yes" for v in row), axis=1) /
        per_study_df[match_cols].shape[1]
    )
    plt.figure(figsize=(6, 4))
    sns.histplot(per_study_df["AgreementRate"], bins=10, kde=False)
    plt.title("Distribution of Agreement Rates per Study")
    plt.xlabel("Agreement Rate")
    plt.ylabel("Number of Studies")
    plt.savefig(f"{output_prefix}_perstudy_hist.png", dpi=300)
    plt.close()

    # --- 4. Trimester Comparison ---
    field = "Pregnancy trimester (1st, 2nd, 3rd, term (for full-term delivery), premature (for early delivery due to complications))"
    if f"{field} — Student" in per_study_df.columns:
        counts = pd.crosstab(per_study_df[f"{field} — Student"], per_study_df[f"{field} — AI"])
        counts.plot(kind="bar", stacked=True, figsize=(8, 5), colormap="tab20")
        plt.title("Trimester Distribution: Student vs AI")
        plt.ylabel("Count")
        plt.xlabel("Student Trimester")
        plt.legend(title="AI Trimester", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        plt.savefig(f"{output_prefix}_trimester.png", dpi=300)
        plt.close()

    print("Figures saved with prefix:", output_prefix)


In [44]:
import matplotlib.pyplot as plt
import seaborn as sns

def make_figures(per_study_df, summary_df, output_prefix="comparison_figs"):
    # --- 1. Agreement Rate by Field ---
    plt.figure(figsize=(8, 10))
    sns.barplot(data=summary_df, y="Field", x="AgreementRate", palette="viridis")
    plt.title("Agreement Rate by Metadata Field")
    plt.xlabel("Agreement Rate")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_agreement_by_field.png", dpi=300)
    plt.close()

    # --- 2. Confusion Matrix for Yes/No ---
    yesno_cols = [c for c in per_study_df.columns if "— Match" in c and "(yes/no)" in c.lower()]
    cm_data = []
    for col in yesno_cols:
        field = col.split(" — ")[0]
        stu_col = f"{field} — Student"
        ai_col  = f"{field} — AI"
        tmp = per_study_df[[stu_col, ai_col]].copy()
        tmp.columns = ["Student", "AI"]
        cm_data.append(tmp)
    if cm_data:
        big = pd.concat(cm_data)
        cm = pd.crosstab(big["Student"], big["AI"])
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title("AI vs Student (Yes/No Fields Combined)")
        plt.savefig(f"{output_prefix}_yesno_confusion.png", dpi=300)
        plt.close()

    # --- 3. Per-Study Agreement Histogram ---
    match_cols = [c for c in per_study_df.columns if "— Match" in c]
    per_study_df["AgreementRate"] = (
        per_study_df[match_cols].apply(lambda row: sum(v=="Yes" for v in row), axis=1) /
        per_study_df[match_cols].shape[1]
    )
    plt.figure(figsize=(6, 4))
    sns.histplot(per_study_df["AgreementRate"], bins=10, kde=False)
    plt.title("Distribution of Agreement Rates per Study")
    plt.xlabel("Agreement Rate")
    plt.ylabel("Number of Studies")
    plt.savefig(f"{output_prefix}_perstudy_hist.png", dpi=300)
    plt.close()

    # --- 4. Trimester Comparison ---
    field = "Pregnancy trimester (1st, 2nd, 3rd, term (for full-term delivery), premature (for early delivery due to complications))"
    if f"{field} — Student" in per_study_df.columns:
        counts = pd.crosstab(per_study_df[f"{field} — Student"], per_study_df[f"{field} — AI"])
        counts.plot(kind="bar", stacked=True, figsize=(8, 5), colormap="tab20")
        plt.title("Trimester Distribution: Student vs AI")
        plt.ylabel("Count")
        plt.xlabel("Student Trimester")
        plt.legend(title="AI Trimester", bbox_to_anchor=(1.05, 1), loc="upper left")
        plt.tight_layout()
        plt.savefig(f"{output_prefix}_trimester.png", dpi=300)
        plt.close()

    print("Figures saved with prefix:", output_prefix)
    return per_study_df, summary_df 


In [47]:
per_study_df, summary_df = main()
make_figures(per_study_df, summary_df)


/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/_read_only.py:81: UserWarning: Cell I400 is marked as a date but the serial value 3698034.0 is outside the limits for dates. The cell will be treated as an error.
  for idx, row in parser.parse():
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/_read_only.py:81: UserWarning: Cell F650 is marked as a date but the serial value 200138762.0 is outside the limits for dates. The cell will be treated as an error.
  for idx, row in parser.parse():
/opt/anaconda3/lib/python3.12/site-packages/openpyxl/worksheet/_read_only.py:81: UserWarning: Cell F744 is marked as a date but the serial value 3698036.0 is outside the limits for dates. The cell will be treated as an error.
  for idx, row in parser.parse():


NameError: name 'merged' is not defined

In [46]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import textwrap

# canonical bins we want
_TRIMESTER_MAP = {
    "1st": {"1", "1st", "first"},
    "2nd": {"2", "2nd", "second"},
    "3rd": {"3", "3rd", "third"},
    "term": {"term", "full-term", "full term"},
    "premature": {"preterm", "premature", "early"}
}

def _wrap_labels(labels, width=38):
    return ["\n".join(textwrap.wrap(str(x), width=width)) for x in labels]

def _norm_yesno(x):
    s = str(x).lower()
    return "Yes" if "yes" in s else "No"

def _norm_trimester(x):
    s = str(x).strip().lower()
    if not s or s in {"nan", "none"}:
        return "unknown/other"
    for canon, keys in _TRIMESTER_MAP.items():
        for k in keys:
            if f" {k} " in f" {s} ":
                return canon
    if s in {"1", "2", "3"}:
        return {"1":"1st","2":"2nd","3":"3rd"}[s]
    return "unknown/other"

def make_figures(per_study_df, summary_df, output_prefix="comparison_figs"):
    # -------- 0) ensure we’re only looking at rows that actually have both sides
    # (Your per_study_df already comes from the inner-join; this is just defensive.)
    df = per_study_df.copy()

    # -------- 1) Agreement Rate by Field (bigger canvas + wrapped labels)
    plt.figure(figsize=(12, max(8, 0.55 * len(summary_df))))
    plot_df = summary_df.copy()
    plot_df = plot_df.sort_values("AgreementRate", ascending=True)
    sns.barplot(data=plot_df, y="Field", x="AgreementRate")
    plt.yticks(ticks=range(len(plot_df)), labels=_wrap_labels(plot_df["Field"].tolist(), width=45), fontsize=9)
    plt.title("Agreement Rate by Metadata Field", fontsize=14, pad=12)
    plt.xlabel("Agreement Rate")
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_agreement_by_field.png", dpi=300)
    plt.close()

    # -------- 2) Yes/No confusion matrix (counts + percentages)
    yesno_match_cols = [
        c for c in df.columns
        if "— Match" in c and "(yes/no)" in c.split(" — ")[0].lower()
    ]
    cm_frames = []
    for mcol in yesno_match_cols:
        base = mcol.split(" — ")[0]
        s_col = f"{base} — Student"
        a_col = f"{base} — AI"
        if s_col in df.columns and a_col in df.columns:
            tmp = df[[s_col, a_col]].copy()
            tmp.columns = ["Student", "AI"]
            tmp["Student"] = tmp["Student"].map(_norm_yesno)
            tmp["AI"]      = tmp["AI"].map(_norm_yesno)
            cm_frames.append(tmp)

    if cm_frames:
        big = pd.concat(cm_frames, ignore_index=True)
        cm_counts = pd.crosstab(big["Student"], big["AI"])
        cm_perc   = cm_counts / cm_counts.to_numpy().sum() * 100.0

        plt.figure(figsize=(8, 6))
        ax = sns.heatmap(cm_counts, annot=True, fmt="d", cbar=False, cmap="Blues")
        ax.set_title("AI vs Student (Yes/No Fields Combined) — Counts", pad=12)
        plt.savefig(f"{output_prefix}_yesno_confusion_counts.png", dpi=300, bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(8, 6))
        ax = sns.heatmap(cm_perc, annot=True, fmt=".1f", cbar=False, cmap="Blues")
        ax.set_title("AI vs Student (Yes/No Fields Combined) — Percent", pad=12)
        plt.savefig(f"{output_prefix}_yesno_confusion_percent.png", dpi=300, bbox_inches="tight")
        plt.close()

    # -------- 3) Per-study agreement histogram (bigger, labeled)
    match_cols = [c for c in df.columns if "— Match" in c]
    if match_cols:
        rates = df[match_cols].apply(lambda r: sum(v == "Yes" for v in r), axis=1) / len(match_cols)
        plt.figure(figsize=(10, 6))
        sns.histplot(rates, bins=12)
        plt.title("Distribution of Agreement Rates per Study", fontsize=16, pad=12)
        plt.xlabel("Agreement Rate")
        plt.ylabel("Number of Studies")
        plt.xlim(0, 1)
        plt.tight_layout()
        plt.savefig(f"{output_prefix}_perstudy_hist.png", dpi=300)
        plt.close()

    # -------- 4) Trimester distribution — normalized and tidy
    field = "Pregnancy trimester (1st, 2nd, 3rd, term (for full-term delivery), premature (for early delivery due to complications))"
    s_col = f"{field} — Student"
    a_col = f"{field} — AI"
    if s_col in df.columns and a_col in df.columns:
        tidy = pd.DataFrame({
            "Student": df[s_col].map(_norm_trimester),
            "AI":      df[a_col].map(_norm_trimester),
        })
        # long form for seaborn
        long = tidy.melt(var_name="Source", value_name="Trimester")

        # counts normalized to percent within each Source
        counts = (long
                  .groupby(["Source", "Trimester"])
                  .size()
                  .groupby(level=0)
                  .apply(lambda s: s / s.sum() * 100)
                  .rename("Percent")
                  .reset_index())

        order = ["1st", "2nd", "3rd", "term", "premature", "unknown/other"]

        plt.figure(figsize=(12, 6))
        sns.barplot(
            data=counts,
            x="Trimester", y="Percent", hue="Source",
            order=order
        )
        plt.title("Trimester Distribution — Student vs AI (Percent)", fontsize=14, pad=10)
        plt.xlabel("")
        plt.ylabel("Percent of Studies")
        plt.ylim(0, 100)
        plt.legend(title="", loc="upper right")
        plt.tight_layout()
        plt.savefig(f"{output_prefix}_trimester_percent.png", dpi=300)
        plt.close()

    print("Figures saved with prefix:", output_prefix)
